In [0]:
from pyspark.sql.functions import col, regexp_replace, replace
from pyspark.sql.functions import avg, first, round, date_diff
from pyspark.sql.functions import broadcast, when, lit
from pyspark.sql import functions as F

##### Write Silver Tables

In [0]:
def write_silver_table(df, table_name, db_name="ecommerce_brazil"):
    # Add metadata information to the table
    added_metadata_df = df.withColumn("_ingested_at", F.current_timestamp()) \
                    .withColumn("_source_file",  lit(table_name))
    
    # set the target path
    target_path = f"{db_name}.silver.{table_name}"
    
    # Write the table with options
    (added_metadata_df.write
        .format("delta")
        .mode("overwrite") 
        .option("mergeSchema", "true")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .saveAsTable(target_path))
    
    print(f"Table {target_path} written successfully.")

##### Customer

In [0]:
# Fetch the customer table from the bronze schema
customer_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.customers")

# Remove duplicates if any from the customer table
customer_df = customer_df.dropDuplicates(["customer_id"])

# Write the customer table to the silver schema
write_silver_table(customer_df, "customers")

##### Geo Location Details

In [0]:
# Fetch the geolocation details table from bronze schema
geo_location_details_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.geolocation_details")

# Aggregate the table as it contains multiple entries for a single zip code 
aggregated_geo_location_details_df = geo_location_details_df.groupBy("geolocation_zip_code_prefix") \
                                            .agg(
                                                avg("geolocation_lat").alias("latitude"), 
                                                avg("geolocation_lng").alias("longitude"),
                                                first("geolocation_city").alias("city"),
                                                first("geolocation_state").alias("state_code"))

In [0]:
# Add an additional column for easier identification of Brazil states
brazil_states = [
    ("AC", "Acre"), ("AL", "Alagoas"), ("AP", "Amapá"), ("AM", "Amazonas"),
    ("BA", "Bahia"), ("CE", "Ceará"), ("DF", "Distrito Federal"), ("ES", "Espírito Santo"),
    ("GO", "Goiás"), ("MA", "Maranhão"), ("MT", "Mato Grosso"), ("MS", "Mato Grosso do Sul"),
    ("MG", "Minas Gerais"), ("PA", "Pará"), ("PB", "Paraíba"), ("PR", "Paraná"),
    ("PE", "Pernambuco"), ("PI", "Piauí"), ("RJ", "Rio de Janeiro"), ("RN", "Rio Grande do Norte"),
    ("RS", "Rio Grande do Sul"), ("RO", "Rondônia"), ("RR", "Roraima"), ("SC", "Santa Catarina"),
    ("SP", "São Paulo"), ("SE", "Sergipe"), ("TO", "Tocantins")
]

# Create a spark dataframe from the above data
state_mapping_df = spark.createDataFrame(brazil_states, ["state_code", "state"])

# Join the aggregated_geo_location_details_df with the state_mapping_df
aggregated_geo_location_details_df = aggregated_geo_location_details_df.join(broadcast(state_mapping_df), on="state_code", how ="left")

# Write the aggregated_geo_location_details_df to the silver schema
write_silver_table(aggregated_geo_location_details_df, "geolocation_details")

##### Order Items

In [0]:
# Fetch the order items table from bronze schema
order_items_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.order_items")

# Add a column to calculate the total va;ue of each order
order_items_df = order_items_df.withColumn("total_value", round(col("price") + col("freight_value"), 2))

# Write the order_items_df to the silver schema
write_silver_table(order_items_df, "order_items")

##### Order Payments

In [0]:
# Fetch the order payments table from bronze schema
order_payments_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.order_payments")

# Filter the rows to inlcude only posiitive payments
order_payments_df = order_payments_df.filter(col("payment_value") > 0)

# Write the order_payments_df to the silver schema
write_silver_table(order_payments_df, "order_payments")

##### Order Reviews

In [0]:
# Fetch the order reviews table from bronze schema
order_reviews_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.order_reviews")

# Filter the rows that includes at least one of review_score, review_comment_title or review_comment_message
order_reviews_df = order_reviews_df.filter(col("review_score").isNotNull() | col("review_comment_title").isNotNull() | col("review_comment_message").isNotNull())

# Handle the line breaks and carriage returns and add additional column to indicate if there are any comments
order_reviews_df = order_reviews_df.withColumn("review_comment_message", regexp_replace(col("review_comment_message"), r"[\n\r]", " ")) \
                                .withColumn("has_review_commnents", col("review_comment_message").isNotNull())

# Write the order_reviews_df to the silver schema
write_silver_table(order_reviews_df, "order_reviews")

##### Orders

In [0]:
# Fetch the orders table from bronze schema
orders_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.orders")

# Add additional columns for frature engineering
orders_df = orders_df.withColumn("delivery_difference_days", date_diff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))

# Write the orders_df to the silver schema
write_silver_table(orders_df, "orders")

##### Product Category Name Translation

In [0]:
# Fetch the product category name translation table from bronze schema
product_category_name_translation_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.product_category_name_translation")

# rename columns and clean data
product_category_name_translation_df = product_category_name_translation_df \
                                            .withColumn("product_category_name_english", regexp_replace(col("product_category_name_english"),"_", " & ")) \
                                            .withColumnRenamed("product_category_name", "category_name_pt") \
                                            .withColumnRenamed("product_category_name_english", "category_name_eng")

##### Products

In [0]:
# Fetch product table from bronze schema
products_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.products")

# Broadcast Join Products and Product Category Name Translation table
products_df = products_df.join(broadcast(product_category_name_translation_df), products_df.product_category_name == product_category_name_translation_df.category_name_pt, "left")

# Clean category name and drop unnecessary columns
products_df = products_df \
    .withColumn("category_name_eng", when(col("category_name_eng").isNull(), "other").otherwise(col("category_name_eng"))) \
    .drop("category_name_pt", "product_category_name")

# Write the products_df to the silver schema
write_silver_table(products_df, "products")

##### Sellers

In [0]:
# Fetch sellers data from bronze schema
sellers_df = spark.sql("SELECT * FROM ecommerce_brazil.bronze.sellers")

# Add state names by joining with state mapping dataframe
sellers_df = sellers_df.join(broadcast(state_mapping_df), sellers_df.seller_state == state_mapping_df.state_code, how ="left").drop("state_code", "seller_state")

# Rename the state column to seller_state
sellers_df = sellers_df.withColumnRenamed("state", "seller_state")

# Write the sellers_df to the silver schema
write_silver_table(sellers_df, "sellers")